# Injini — train & evaluate

DCASE Task 2 first-shot recipe compressed for an Arm phone: a frozen
AudioSet-distilled MobileNetV3 embedder plus a Mahalanobis distance score.
This notebook is self-contained. It writes the pipeline, pulls EfficientAT and
the DCASE dev set, runs the full evaluation, and exports the model artefacts.

**Settings:** Internet ON. GPU (P100 or T4). Add data: `zeyadzsm/engine-sounds`.
Then Run All.

## 1. Workspace and pipeline files

In [ ]:

import os, sys, shutil, json, glob, time, subprocess
WORK = "/kaggle/working/injini"
os.makedirs(WORK + "/src", exist_ok=True)
os.makedirs(WORK + "/export", exist_ok=True)
os.makedirs(WORK + "/models", exist_ok=True)
os.makedirs(WORK + "/data", exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, WORK + "/src")
print("workspace", WORK)

In [ ]:
%%writefile src/features.py
"""Log-mel front end for Injini.

The embedder Injini ships is an EfficientAT MobileNetV3 pretrained on AudioSet.
Those weights are only valid against EfficientAT's own mel spec, so this module
reproduces that spec exactly:

    sample rate 32 kHz, pre-emphasis 0.97, STFT n_fft=1024 / hop=320 /
    win_length=800 (symmetric Hann, zero-padded to n_fft, centered/reflect),
    128 Kaldi-style mel bands 0-15000 Hz, log(mel + 1e-5), then (x + 4.5) / 5.

Two implementations of that one spec:

  * ``reference_logmel`` uses EfficientAT's ``AugmentMelSTFT`` in eval mode. This
    is the ground truth used for training and evaluation on Kaggle.
  * ``logmel`` is a pure-NumPy reimplementation with no torch / torchaudio at
    run time. It loads a precomputed Kaldi mel matrix (``models/mel_kaldi_128x513.npy``,
    written by ``dump_mel_matrix``) so the fiddly Kaldi filterbank never has to
    be re-derived. This is the version ported to Kotlin for the phone, and
    ``tests/test_feature_parity.py`` holds the two within 1e-3.

Capture on the phone is 16 kHz (matches SiloSense, matches the Android
UNPROCESSED source, keeps knock detection honestly out of scope). ``load_audio``
resamples whatever it is given to 32 kHz with the same naive linear interp
SiloSense used.
"""
from __future__ import annotations

import os

import numpy as np
import soundfile as sf

SR = 32_000
N_FFT = 1024
HOP = 320
WIN_LENGTH = 800
N_MELS = 128
FMIN = 0.0
FMAX = 15_000.0  # AugmentMelSTFT eval: sr // 2 - fmax_aug_range // 2 = 16000 - 1000
PREEMPH = 0.97
LOG_OFFSET = 1e-5
NORM_ADD = 4.5
NORM_DIV = 5.0

CLIP_SECONDS = 10.0
CLIP_SAMPLES = int(SR * CLIP_SECONDS)
N_FRAMES = 1 + CLIP_SAMPLES // HOP  # 1001, the fixed length used for the on-device ONNX graph

_HERE = os.path.dirname(os.path.abspath(__file__))
MEL_MATRIX_PATH = os.path.join(_HERE, os.pardir, "models", "mel_kaldi_128x513.npy")

# Symmetric (periodic=False) Hann of win_length, zero-padded to n_fft and centered,
# matching torch.stft(win_length=800, n_fft=1024, window=hann_window(800, periodic=False)).
_hann = 0.5 - 0.5 * np.cos(2.0 * np.pi * np.arange(WIN_LENGTH) / (WIN_LENGTH - 1))
_WINDOW = np.zeros(N_FFT, dtype=np.float64)
_pad_left = (N_FFT - WIN_LENGTH) // 2
_WINDOW[_pad_left:_pad_left + WIN_LENGTH] = _hann

_MEL: np.ndarray | None = None


def dump_mel_matrix(path: str = MEL_MATRIX_PATH) -> np.ndarray:
    """Compute EfficientAT's Kaldi mel filterbank once (needs torchaudio) and cache it.

    Returns a (128, 513) float32 matrix: get_mel_banks(...) padded with one zero
    column, exactly as AugmentMelSTFT does before ``mel_basis @ power``.
    """
    import torch
    import torchaudio

    mel_basis, _ = torchaudio.compliance.kaldi.get_mel_banks(
        N_MELS, N_FFT, SR, FMIN, FMAX,
        vtln_low=100.0, vtln_high=-500.0, vtln_warp_factor=1.0,
    )
    mel_basis = torch.nn.functional.pad(mel_basis, (0, 1), mode="constant", value=0)
    mat = mel_basis.cpu().numpy().astype(np.float32)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    np.save(path, mat)
    return mat


def mel_matrix() -> np.ndarray:
    global _MEL
    if _MEL is None:
        if not os.path.exists(MEL_MATRIX_PATH):
            dump_mel_matrix(MEL_MATRIX_PATH)
        _MEL = np.load(MEL_MATRIX_PATH).astype(np.float64)
    return _MEL


def resample_linear(y: np.ndarray, orig_sr: int, target_sr: int = SR) -> np.ndarray:
    if orig_sr == target_sr or len(y) == 0:
        return y.astype(np.float32)
    duration = len(y) / orig_sr
    n_target = max(1, int(round(duration * target_sr)))
    x_orig = np.linspace(0.0, duration, num=len(y), endpoint=False)
    x_target = np.linspace(0.0, duration, num=n_target, endpoint=False)
    return np.interp(x_target, x_orig, y).astype(np.float32)


def load_audio(path: str, offset: float = 0.0, duration: float | None = None) -> np.ndarray:
    with sf.SoundFile(path) as f:
        sr = f.samplerate
        f.seek(int(offset * sr))
        n = int(duration * sr) if duration is not None else -1
        y = f.read(frames=n, dtype="float32", always_2d=False)
    if y.ndim > 1:
        y = y.mean(axis=1)
    return resample_linear(np.asarray(y, dtype=np.float32), sr, SR)


def fixed_length(y: np.ndarray, n: int = CLIP_SAMPLES) -> np.ndarray:
    if len(y) >= n:
        return y[:n]
    return np.pad(y, (0, n - len(y)), mode="constant")


def _preemphasis(y: np.ndarray) -> np.ndarray:
    # torch: conv1d(x, [[[-.97, 1]]]) -> out[t] = -0.97*x[t] + x[t+1], length N-1.
    return (y[1:] - PREEMPH * y[:-1]).astype(np.float64)


def _stft_power(y: np.ndarray) -> np.ndarray:
    pad = N_FFT // 2
    yp = np.pad(y, (pad, pad), mode="reflect")
    n_frames = 1 + (len(yp) - N_FFT) // HOP
    idx = np.arange(N_FFT)[:, None] + HOP * np.arange(n_frames)[None, :]
    frames = yp[idx] * _WINDOW[:, None]
    spec = np.fft.rfft(frames, n=N_FFT, axis=0)
    return (spec.real ** 2 + spec.imag ** 2)  # (513, n_frames)


def logmel(y: np.ndarray, fixed: bool = False) -> np.ndarray:
    """Pure-NumPy log-mel matching ``reference_logmel``. (128, T) float32.

    ``fixed=True`` pins the output to ``N_FRAMES`` columns for the on-device graph.
    """
    if fixed:
        y = fixed_length(y)
    power = _stft_power(_preemphasis(np.asarray(y, dtype=np.float64)))
    mel = mel_matrix() @ power
    mel = np.log(mel + LOG_OFFSET)
    mel = (mel + NORM_ADD) / NORM_DIV
    out = mel.astype(np.float32)
    if fixed:
        if out.shape[1] >= N_FRAMES:
            out = out[:, :N_FRAMES]
        else:
            out = np.pad(out, ((0, 0), (0, N_FRAMES - out.shape[1])), mode="edge")
    return out


def reference_logmel(y: np.ndarray) -> np.ndarray:
    """EfficientAT AugmentMelSTFT in eval mode. Ground truth. Needs torch/torchaudio."""
    import torch
    import sys

    vendor = os.path.join(_HERE, os.pardir, "vendor_efficientat")
    if vendor not in sys.path:
        sys.path.insert(0, vendor)
    from models.preprocess import AugmentMelSTFT

    mel = AugmentMelSTFT(
        n_mels=N_MELS, sr=SR, win_length=WIN_LENGTH, hopsize=HOP, n_fft=N_FFT,
        freqm=0, timem=0, fmin=FMIN, fmax=FMAX, fmin_aug_range=1, fmax_aug_range=1,
    )
    mel.eval()
    with torch.no_grad():
        spec = mel(torch.from_numpy(np.asarray(y, dtype=np.float32))[None, :])
    return spec.squeeze(0).cpu().numpy()


if __name__ == "__main__":
    mat = dump_mel_matrix()
    print(f"wrote {MEL_MATRIX_PATH}  shape={mat.shape}  sum={mat.sum():.3f}")

In [ ]:
%%writefile src/metrics.py
"""DCASE Task 2 scoring: AUC, partial AUC (p=0.1), and the official harmonic-mean score."""
from __future__ import annotations

import numpy as np
from sklearn.metrics import roc_auc_score

P = 0.1


def partial_auc(y_true: np.ndarray, y_score: np.ndarray, p: float = P) -> float:
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    neg = y_score[y_true == 0]
    pos = y_score[y_true == 1]
    if len(neg) == 0 or len(pos) == 0:
        return float("nan")
    n_keep = max(1, int(np.ceil(p * len(neg))))
    thr = np.sort(neg)[::-1][n_keep - 1]
    kept_neg = neg[neg >= thr]
    # fraction of positives ranked above each kept negative, averaged
    return float(np.mean([(pos > t).mean() + 0.5 * (pos == t).mean() for t in kept_neg]))


def auc(y_true: np.ndarray, y_score: np.ndarray) -> float:
    y_true = np.asarray(y_true)
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, y_score))


def harmonic_mean(values: list[float]) -> float:
    v = np.asarray([x for x in values if x == x and x > 0], dtype=np.float64)
    if len(v) == 0:
        return float("nan")
    return float(len(v) / np.sum(1.0 / v))


def official_score(per_machine: dict[str, dict]) -> float:
    """per_machine[name] = {'auc_source':.., 'auc_target':.., 'pauc':..}."""
    pieces: list[float] = []
    for m in per_machine.values():
        pieces += [m.get("auc_source", np.nan), m.get("auc_target", np.nan), m.get("pauc", np.nan)]
    return harmonic_mean(pieces)

In [ ]:
%%writefile src/anomaly.py
"""Distance-based anomaly scoring in embedding space.

This is the whole decision layer. Given the embeddings of a machine's healthy
recordings, score a new clip by how far it sits from them. Two scores, both
standard in DCASE Task 2 systems:

  * ``MahalanobisScorer`` fits a mean and a shrinkage covariance (Ledoit-Wolf)
    on the healthy embeddings and returns the Mahalanobis distance. When source
    and target healthy sets are both given it keeps a covariance for each and
    takes the smaller distance, matching the DCASE "selective Mahalanobis"
    baseline.
  * ``KnnScorer`` returns the mean cosine distance to the k nearest healthy
    embeddings.

Everything here is a few matrix ops on the CPU. It is never exported to ONNX.
On the phone the fitted statistics are what "enrollment" produces and stores.
"""
from __future__ import annotations

import numpy as np

try:
    from sklearn.covariance import LedoitWolf
except Exception:  # pragma: no cover
    LedoitWolf = None


def _fit_gaussian(x: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    mu = x.mean(axis=0)
    xc = x - mu
    if LedoitWolf is not None and len(x) > 2:
        cov = LedoitWolf().fit(xc).covariance_
    else:
        cov = np.cov(xc, rowvar=False) + 1e-6 * np.eye(x.shape[1])
    inv = np.linalg.pinv(cov)
    return mu.astype(np.float64), inv.astype(np.float64)


class MahalanobisScorer:
    def __init__(self) -> None:
        self.stats: dict[str, tuple[np.ndarray, np.ndarray]] = {}

    def fit(self, source: np.ndarray, target: np.ndarray | None = None) -> "MahalanobisScorer":
        self.stats["source"] = _fit_gaussian(np.asarray(source, dtype=np.float64))
        if target is not None and len(target) >= 2:
            self.stats["target"] = _fit_gaussian(np.asarray(target, dtype=np.float64))
        return self

    def score(self, x: np.ndarray) -> np.ndarray:
        x = np.atleast_2d(np.asarray(x, dtype=np.float64))
        dists = []
        for mu, inv in self.stats.values():
            xc = x - mu
            dists.append(np.sqrt(np.einsum("ij,jk,ik->i", xc, inv, xc)))
        return np.min(np.stack(dists, axis=0), axis=0)

    def to_dict(self) -> dict:
        return {k: {"mean": mu.tolist(), "inv_cov": inv.tolist()} for k, (mu, inv) in self.stats.items()}

    @classmethod
    def from_dict(cls, d: dict) -> "MahalanobisScorer":
        s = cls()
        s.stats = {k: (np.array(v["mean"]), np.array(v["inv_cov"])) for k, v in d.items()}
        return s


class KnnScorer:
    def __init__(self, k: int = 4) -> None:
        self.k = k
        self.ref: np.ndarray | None = None

    def fit(self, source: np.ndarray, target: np.ndarray | None = None) -> "KnnScorer":
        ref = np.asarray(source, dtype=np.float64)
        if target is not None and len(target):
            ref = np.vstack([ref, np.asarray(target, dtype=np.float64)])
        self.ref = ref / (np.linalg.norm(ref, axis=1, keepdims=True) + 1e-12)
        return self

    def score(self, x: np.ndarray) -> np.ndarray:
        assert self.ref is not None
        x = np.atleast_2d(np.asarray(x, dtype=np.float64))
        x = x / (np.linalg.norm(x, axis=1, keepdims=True) + 1e-12)
        sim = x @ self.ref.T
        k = min(self.k, self.ref.shape[0])
        topk = np.partition(sim, -k, axis=1)[:, -k:]
        return 1.0 - topk.mean(axis=1)


SCORERS = {"maha": MahalanobisScorer, "knn": KnnScorer}

In [ ]:
%%writefile src/dcase_data.py
"""DCASE 2025 Task 2 (development set) directory reader.

Expected layout under ``--root`` (as distributed):

    <root>/<machine>/train/section_00_source_train_normal_0001_<attrs>.wav
    <root>/<machine>/test/section_00_source_test_normal_0001_<attrs>.wav
    <root>/<machine>/test/section_00_target_test_anomaly_0002_<attrs>.wav

Train is normal only. Test mixes normal and anomaly across source and target
domains. Machine types in the 2025 dev set: ToyCar, ToyTrain, bearing, fan,
gearbox, slider, valve.
"""
from __future__ import annotations

import glob
import os
from dataclasses import dataclass

DEV_MACHINES = ["ToyCar", "ToyTrain", "bearing", "fan", "gearbox", "slider", "valve"]


@dataclass
class Clip:
    path: str
    machine: str
    section: str
    domain: str          # "source" | "target"
    split: str           # "train" | "test"
    label: int           # 0 normal, 1 anomaly


def _parse(path: str, machine: str) -> Clip | None:
    base = os.path.basename(path).lower()
    if not base.endswith(".wav"):
        return None
    parts = base.split("_")
    try:
        section = "section_" + parts[1]
    except IndexError:
        section = "section_00"
    domain = "target" if "target" in base else "source"
    split = "train" if "train" in base else "test"
    label = 1 if "anomaly" in base else 0
    return Clip(path, machine, section, domain, split, label)


def load(root: str, machines: list[str] | None = None) -> list[Clip]:
    machines = machines or DEV_MACHINES
    clips: list[Clip] = []
    for m in machines:
        for sub in ("train", "test"):
            for p in sorted(glob.glob(os.path.join(root, m, sub, "*.wav"))):
                c = _parse(p, m)
                if c is not None:
                    clips.append(c)
    return clips


def by_machine(clips: list[Clip]) -> dict[str, list[Clip]]:
    out: dict[str, list[Clip]] = {}
    for c in clips:
        out.setdefault(c.machine, []).append(c)
    return out

In [ ]:
%%writefile src/embedder.py
"""Frozen EfficientAT MobileNetV3 embedder.

The state-of-the-art DCASE Task 2 recipe scores a clip by its distance, in the
embedding space of a large frozen audio network, from the healthy recordings of
that machine. Injini keeps the recipe and swaps the network for a MobileNetV3
distilled from a transformer teacher on AudioSet (EfficientAT ``mn10_as`` /
``mn04_as``), which is small enough to quantise onto an Arm phone.

This module wraps EfficientAT's ``MN`` so ``forward`` returns only the
L2-normalised embedding (``F.adaptive_avg_pool2d`` of the last feature map), and
exports that subgraph to ONNX. The anomaly score is computed outside the graph
(see ``anomaly.py``); it is a few lines of linear algebra and never needs Arm
acceleration.
"""
from __future__ import annotations

import contextlib
import io
import os
import sys

import torch
import torch.nn as nn
import torch.nn.functional as F

_HERE = os.path.dirname(os.path.abspath(__file__))
VENDOR = os.path.join(_HERE, os.pardir, "vendor_efficientat")

WIDTH = {"mn10_as": 1.0, "mn04_as": 0.4, "mn05_as": 0.5, "mn01_as": 0.1}
EMBED_DIM = {"mn10_as": 960, "mn04_as": 384, "mn05_as": 480, "mn01_as": 96}


def _load_mn(name: str) -> nn.Module:
    # vendor_efficientat/helpers/utils.py reads metadata/class_labels_indices.csv
    # with a cwd-relative path at import time, so run the import from VENDOR.
    if VENDOR not in sys.path:
        sys.path.insert(0, VENDOR)
    cwd = os.getcwd()
    try:
        os.chdir(VENDOR)
        from models.mn.model import get_model

        with contextlib.redirect_stdout(io.StringIO()):
            model = get_model(pretrained_name=name, width_mult=WIDTH[name], head_type="mlp")
    finally:
        os.chdir(cwd)
    return model.eval()


class Embedder(nn.Module):
    """(B, 1, 128, T) log-mel  ->  (B, D) L2-normalised embedding."""

    def __init__(self, name: str = "mn10_as", l2: bool = True):
        super().__init__()
        self.name = name
        self.l2 = l2
        self.features = _load_mn(name).features
        for p in self.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = F.adaptive_avg_pool2d(x, (1, 1)).flatten(1)
        if self.l2:
            x = F.normalize(x, dim=1)
        return x


def export_onnx(name: str, out_path: str, opset: int = 17) -> str:
    from features import N_FRAMES, N_MELS

    model = Embedder(name)
    dummy = torch.randn(1, 1, N_MELS, N_FRAMES)
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    torch.onnx.export(
        model, dummy, out_path,
        input_names=["logmel"], output_names=["embedding"],
        dynamic_axes={"logmel": {0: "batch", 3: "frames"}, "embedding": {0: "batch"}},
        opset_version=opset, dynamo=False,
    )
    return out_path


def param_counts(name: str) -> dict:
    m = Embedder(name)
    total = sum(p.numel() for p in m.parameters())
    return {"embedder": name, "embed_dim": EMBED_DIM[name], "params": total}


if __name__ == "__main__":
    import argparse

    sys.path.insert(0, _HERE)
    ap = argparse.ArgumentParser()
    ap.add_argument("--name", default="mn10_as", choices=list(WIDTH))
    ap.add_argument("--out", default=None)
    args = ap.parse_args()
    out = args.out or os.path.join(_HERE, os.pardir, "models", f"injini_{args.name}_fp32.onnx")
    export_onnx(args.name, out)
    size = os.path.getsize(out) / 1e6
    print({**param_counts(args.name), "onnx_mb": round(size, 3), "path": out})

In [ ]:
%%writefile src/embed_backends.py
"""Uniform embedding interface over the three backends the eval compares.

  * ``torch:<name>``   frozen EfficientAT MobileNetV3 in PyTorch (mn10_as / mn04_as)
  * ``onnx:<path>``    an exported / quantised embedder run through onnxruntime
  * ``passt``          the transformer teacher, the full-size reference ceiling
                       (needs ``hear21passt``; only used on Kaggle)

All backends take a list of waveforms at ``features.SR`` and return an
(N, D) float32 array of L2-normalised embeddings.
"""
from __future__ import annotations

import os
import sys

import numpy as np

_HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, _HERE)

import features as F  # noqa: E402


def _batched_logmels(waves: list[np.ndarray]) -> np.ndarray:
    mels = [F.logmel(w, fixed=True) for w in waves]
    return np.stack(mels, axis=0)[:, None, :, :].astype(np.float32)  # (N,1,128,T)


class TorchBackend:
    def __init__(self, name: str):
        import torch
        from embedder import Embedder

        self.torch = torch
        self.model = Embedder(name)

    def embed(self, waves: list[np.ndarray], batch: int = 16) -> np.ndarray:
        x = _batched_logmels(waves)
        out = []
        for i in range(0, len(x), batch):
            t = self.torch.from_numpy(x[i:i + batch])
            with self.torch.no_grad():
                out.append(self.model(t).cpu().numpy())
        return np.concatenate(out, axis=0).astype(np.float32)


class OnnxBackend:
    def __init__(self, path: str, providers: list[str] | None = None):
        import onnxruntime as ort

        providers = providers or ["CPUExecutionProvider"]
        self.sess = ort.InferenceSession(path, providers=providers)
        self.iname = self.sess.get_inputs()[0].name

    def embed(self, waves: list[np.ndarray], batch: int = 16) -> np.ndarray:
        x = _batched_logmels(waves)
        out = []
        for i in range(0, len(x), batch):
            out.append(self.sess.run(None, {self.iname: x[i:i + batch]})[0])
        e = np.concatenate(out, axis=0).astype(np.float32)
        return e / (np.linalg.norm(e, axis=1, keepdims=True) + 1e-12)


class PasstBackend:
    def __init__(self):
        from hear21passt.base import load_model, get_scene_embeddings

        self.model = load_model(mode="embed_only").eval()
        self._embed = get_scene_embeddings
        import torch

        self.torch = torch

    def embed(self, waves: list[np.ndarray], batch: int = 8) -> np.ndarray:
        out = []
        for i in range(0, len(waves), batch):
            chunk = waves[i:i + batch]
            n = max(len(w) for w in chunk)
            arr = np.zeros((len(chunk), n), dtype=np.float32)
            for j, w in enumerate(chunk):
                arr[j, : len(w)] = w
            t = self.torch.from_numpy(arr)
            with self.torch.no_grad():
                out.append(self._embed(t, self.model).cpu().numpy())
        e = np.concatenate(out, axis=0).astype(np.float32)
        return e / (np.linalg.norm(e, axis=1, keepdims=True) + 1e-12)


def get_backend(spec: str):
    if spec == "passt":
        return PasstBackend()
    if spec.startswith("onnx:"):
        return OnnxBackend(spec.split(":", 1)[1])
    if spec.startswith("torch:"):
        return TorchBackend(spec.split(":", 1)[1])
    raise ValueError(f"unknown backend spec: {spec!r}")

In [ ]:
%%writefile src/eval_dcase.py
"""Evaluate an embedder + distance scorer on the DCASE 2025 Task 2 dev set.

For each machine: embed the training clips (normal only), fit the scorer on the
source and target healthy embeddings, score every test clip, then compute
source AUC, target AUC and partial AUC. The official score is the harmonic mean
of all three across all machines.

    python src/eval_dcase.py --root data/dcase2025_dev --backend torch:mn10_as --scorer maha
    python src/eval_dcase.py --root data/dcase2025_dev --backend onnx:models/injini_mn10_as_int8.onnx

Writes a metrics JSON to models/eval_<tag>.json.
"""
from __future__ import annotations

import argparse
import json
import os
import sys
import time

import numpy as np

_HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, _HERE)

import features as F          # noqa: E402
import dcase_data as D        # noqa: E402
import metrics as M           # noqa: E402
from anomaly import SCORERS   # noqa: E402
from embed_backends import get_backend  # noqa: E402


def _load(paths: list[str]) -> list[np.ndarray]:
    return [F.load_audio(p) for p in paths]


def evaluate(root: str, backend_spec: str, scorer_name: str, machines=None, limit=None) -> dict:
    backend = get_backend(backend_spec)
    clips = D.load(root, machines)
    if not clips:
        raise SystemExit(f"no clips under {root}")
    per_machine: dict[str, dict] = {}
    t0 = time.time()

    for machine, mclips in D.by_machine(clips).items():
        train = [c for c in mclips if c.split == "train"]
        test = [c for c in mclips if c.split == "test"]
        if limit:
            train, test = train[:limit], test[:limit]
        if not train or not test:
            continue

        tr_emb = backend.embed(_load([c.path for c in train]))
        tr_src = tr_emb[[i for i, c in enumerate(train) if c.domain == "source"]]
        tr_tgt = tr_emb[[i for i, c in enumerate(train) if c.domain == "target"]]

        scorer = SCORERS[scorer_name]()
        scorer.fit(tr_src if len(tr_src) else tr_emb,
                   tr_tgt if len(tr_tgt) >= 2 else None)

        te_emb = backend.embed(_load([c.path for c in test]))
        scores = scorer.score(te_emb)
        y = np.array([c.label for c in test])
        dom = np.array([c.domain for c in test])

        src = dom == "source"
        tgt = dom == "target"
        per_machine[machine] = {
            "auc_source": M.auc(y[src], scores[src]) if src.any() else float("nan"),
            "auc_target": M.auc(y[tgt], scores[tgt]) if tgt.any() else float("nan"),
            "pauc": M.partial_auc(y, scores),
            "n_train": len(train), "n_test": len(test),
        }
        print(f"  {machine:9s} "
              f"AUC_src={per_machine[machine]['auc_source']:.4f} "
              f"AUC_tgt={per_machine[machine]['auc_target']:.4f} "
              f"pAUC={per_machine[machine]['pauc']:.4f}")

    result = {
        "backend": backend_spec,
        "scorer": scorer_name,
        "official_score": M.official_score(per_machine),
        "mean_auc": float(np.nanmean([v["auc_source"] for v in per_machine.values()]
                                     + [v["auc_target"] for v in per_machine.values()])),
        "per_machine": per_machine,
        "seconds": round(time.time() - t0, 1),
    }
    return result


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", required=True)
    ap.add_argument("--backend", default="torch:mn10_as")
    ap.add_argument("--scorer", default="maha", choices=list(SCORERS))
    ap.add_argument("--machines", nargs="*", default=None)
    ap.add_argument("--limit", type=int, default=None, help="cap clips per split (smoke test)")
    ap.add_argument("--out", default=None)
    args = ap.parse_args()

    res = evaluate(args.root, args.backend, args.scorer, args.machines, args.limit)
    print(json.dumps({k: v for k, v in res.items() if k != "per_machine"}, indent=2))

    tag = args.backend.replace(":", "_").replace("/", "_").replace("\\", "_").replace(".onnx", "")
    out = args.out or os.path.join(_HERE, os.pardir, "models", f"eval_{tag}_{args.scorer}.json")
    with open(out, "w") as f:
        json.dump(res, f, indent=2)
    print("wrote", out)

In [ ]:
%%writefile src/baseline_ae.py
"""The DCASE Task 2 autoencoder baseline, reproduced.

Identical in shape to the 2023-2025 official baseline: log-mel with 128 bands,
5 consecutive frames concatenated into each 640-d input vector, a dense
encoder/decoder, trained to minimise reconstruction MSE on normal sound only.
Two anomaly scores:

  * "mse"   : mean reconstruction error over a clip's frames
  * "maha"  : Mahalanobis distance between input and reconstruction under a
              covariance fitted on the training residuals (selective, source
              vs target, smaller distance kept)

This exists so Section 10 can show we reproduce the published baseline before
claiming anything about beating it. Runs on the same DCASE dev tree as
eval_dcase.py.

    python src/baseline_ae.py --root data/dcase2025_dev --epochs 100
"""
from __future__ import annotations

import argparse
import json
import os
import sys

import numpy as np
import torch
import torch.nn as nn

_HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, _HERE)

import features as F        # noqa: E402
import dcase_data as D      # noqa: E402
import metrics as M         # noqa: E402

CTX = 5
IN_DIM = F.N_MELS * CTX


class DenseAE(nn.Module):
    def __init__(self, in_dim: int = IN_DIM):
        super().__init__()
        h = 128
        self.enc = nn.Sequential(
            nn.Linear(in_dim, h), nn.BatchNorm1d(h), nn.ReLU(),
            nn.Linear(h, h), nn.BatchNorm1d(h), nn.ReLU(),
            nn.Linear(h, h), nn.BatchNorm1d(h), nn.ReLU(),
            nn.Linear(h, h), nn.BatchNorm1d(h), nn.ReLU(),
            nn.Linear(h, 8), nn.BatchNorm1d(8), nn.ReLU(),
        )
        self.dec = nn.Sequential(
            nn.Linear(8, h), nn.BatchNorm1d(h), nn.ReLU(),
            nn.Linear(h, h), nn.BatchNorm1d(h), nn.ReLU(),
            nn.Linear(h, h), nn.BatchNorm1d(h), nn.ReLU(),
            nn.Linear(h, h), nn.BatchNorm1d(h), nn.ReLU(),
            nn.Linear(h, in_dim),
        )

    def forward(self, x):
        return self.dec(self.enc(x))


def _frames(path: str) -> np.ndarray:
    mel = F.logmel(F.load_audio(path))            # (128, T)
    T = mel.shape[1]
    if T < CTX:
        mel = np.pad(mel, ((0, 0), (0, CTX - T)), mode="edge")
        T = CTX
    idx = np.arange(CTX)[None, :] + np.arange(T - CTX + 1)[:, None]
    return mel[:, idx].transpose(1, 0, 2).reshape(T - CTX + 1, -1).astype(np.float32)


def train_machine(train_paths, epochs, device):
    X = np.concatenate([_frames(p) for p in train_paths], axis=0)
    ds = torch.utils.data.TensorDataset(torch.from_numpy(X))
    dl = torch.utils.data.DataLoader(ds, batch_size=512, shuffle=True, drop_last=True)
    net = DenseAE().to(device)
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    lossf = nn.MSELoss()
    for _ in range(epochs):
        net.train()
        for (xb,) in dl:
            xb = xb.to(device)
            opt.zero_grad()
            loss = lossf(net(xb), xb)
            loss.backward()
            opt.step()
    return net


@torch.no_grad()
def clip_score(net, path, device):
    net.eval()
    x = torch.from_numpy(_frames(path)).to(device)
    r = net(x)
    return float(((x - r) ** 2).mean().item())


def evaluate(root, epochs=100, machines=None, limit=None):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    clips = D.load(root, machines)
    per_machine = {}
    for machine, mclips in D.by_machine(clips).items():
        train = [c for c in mclips if c.split == "train"]
        test = [c for c in mclips if c.split == "test"]
        if limit:
            train, test = train[:limit], test[:limit]
        if not train or not test:
            continue
        net = train_machine([c.path for c in train], epochs, device)
        scores = np.array([clip_score(net, c.path, device) for c in test])
        y = np.array([c.label for c in test])
        dom = np.array([c.domain for c in test])
        per_machine[machine] = {
            "auc_source": M.auc(y[dom == "source"], scores[dom == "source"]),
            "auc_target": M.auc(y[dom == "target"], scores[dom == "target"]),
            "pauc": M.partial_auc(y, scores),
        }
        print(f"  {machine:9s} {per_machine[machine]}")
    return {
        "system": "dcase_ae_baseline_mse",
        "epochs": epochs,
        "official_score": M.official_score(per_machine),
        "per_machine": per_machine,
    }


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", required=True)
    ap.add_argument("--epochs", type=int, default=100)
    ap.add_argument("--machines", nargs="*", default=None)
    ap.add_argument("--limit", type=int, default=None)
    args = ap.parse_args()
    res = evaluate(args.root, args.epochs, args.machines, args.limit)
    print(json.dumps({k: v for k, v in res.items() if k != "per_machine"}, indent=2))
    out = os.path.join(_HERE, os.pardir, "models", "eval_dcase_ae_baseline.json")
    with open(out, "w") as f:
        json.dump(res, f, indent=2)
    print("wrote", out)

In [ ]:
%%writefile src/fetch_dcase.py
"""Download and unpack the DCASE 2025 Task 2 development set (Zenodo 15097779).

CC BY-NC-SA 4.0. Seven machine-type zips, ~2.3 GB total. Unpacks to
<out>/<machine>/{train,test}/*.wav, the layout dcase_data.py expects.

    python src/fetch_dcase.py --out data/dcase2025_dev
    python src/fetch_dcase.py --out data/dcase2025_dev --machines bearing fan
"""
from __future__ import annotations

import argparse
import os
import urllib.request
import zipfile

RECORD = "https://zenodo.org/records/15097779/files"
MACHINES = ["bearing", "fan", "gearbox", "slider", "valve", "ToyCar", "ToyTrain"]


def fetch(out: str, machines: list[str]) -> None:
    os.makedirs(out, exist_ok=True)
    for m in machines:
        zpath = os.path.join(out, f"dev_{m}.zip")
        if not os.path.exists(zpath):
            url = f"{RECORD}/dev_{m}.zip?download=1"
            print(f"downloading {url}")
            urllib.request.urlretrieve(url, zpath)
        print(f"unpacking {zpath}")
        with zipfile.ZipFile(zpath) as z:
            z.extractall(out)
    # Zenodo zips unpack as <out>/<machine>/{train,test}; some editions nest an
    # extra top folder. Flatten if needed.
    for m in machines:
        nested = os.path.join(out, m, m)
        if os.path.isdir(nested):
            for sub in os.listdir(nested):
                os.replace(os.path.join(nested, sub), os.path.join(out, m, sub))


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default="data/dcase2025_dev")
    ap.add_argument("--machines", nargs="*", default=MACHINES)
    args = ap.parse_args()
    fetch(args.out, args.machines)
    print("done")

In [ ]:
%%writefile src/prepare_engine_sounds.py
"""Turn the Kaggle 'zeyadzsm/engine-sounds' dump into a source-split manifest.

Two problems with that dataset, both handled here:

1. It ships augmented variants next to their source clips, with names like
   ``1_augmented_10_Alternator Bearing Noise.wav``, ``1_segment_0_augmented.wav``,
   ``002_Engine-A_augmented_1.wav``. Splitting those at file level leaks
   near-identical audio across train and test. Every file is reduced to a
   ``source key`` (strip augmentation / segment / trailing-index suffixes) and
   the split is done on the (class, source key) pair.
2. Two near-duplicate top folders (``Data/Data_Fixed`` and
   ``Data_AA/Data_Fixed``). Both are read; the source key is taken relative to
   the class folder so a clip appearing in both maps to one key.

Output: data/engine_sounds_manifest.csv with columns
    path, label, class_name, source_key, split
"""
from __future__ import annotations

import argparse
import csv
import glob
import os
import random
import re

AUG_PATTERNS = [
    r"_augmented(_\d+)?(_[A-Za-z].*)?$",
    r"_segment_\d+(_augmented.*)?$",
    r"_aug(_\d+)?$",
]
TRAIL_IDX = re.compile(r"(_\d+)+$")


def source_key(stem: str) -> str:
    s = stem
    for pat in AUG_PATTERNS:
        s = re.sub(pat, "", s, flags=re.IGNORECASE)
    s = TRAIL_IDX.sub("", s)
    return s.strip() or stem


def collect(root: str) -> list[dict]:
    rows = []
    classes = set()
    for wav in glob.glob(os.path.join(root, "**", "*.wav"), recursive=True):
        rel = os.path.relpath(wav, root).replace("\\", "/")
        parts = rel.split("/")
        # .../<something>/Data_Fixed/<class>/<maybe Augmented>/<file>.wav
        try:
            di = parts.index("Data_Fixed")
            cls = parts[di + 1]
        except (ValueError, IndexError):
            cls = parts[-2]
        classes.add(cls)
        stem = os.path.splitext(parts[-1])[0]
        rows.append({"path": wav, "class_name": cls, "source_key": f"{cls}::{source_key(stem)}"})
    labels = {c: i for i, c in enumerate(sorted(classes))}
    for r in rows:
        r["label"] = labels[r["class_name"]]
    return rows


def split(rows: list[dict], val_frac=0.2, seed=42) -> list[dict]:
    rng = random.Random(seed)
    keys_by_class: dict[str, list[str]] = {}
    for r in rows:
        keys_by_class.setdefault(r["class_name"], set()).add(r["source_key"])
    val_keys: set[str] = set()
    for cls, keys in keys_by_class.items():
        keys = sorted(keys)
        rng.shuffle(keys)
        n_val = max(1, round(len(keys) * val_frac))
        val_keys.update(keys[:n_val])
    for r in rows:
        r["split"] = "val" if r["source_key"] in val_keys else "train"
    return rows


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", required=True, help="folder containing Data/ and Data_AA/")
    ap.add_argument("--out", default="data/engine_sounds_manifest.csv")
    args = ap.parse_args()

    rows = split(collect(args.root))
    os.makedirs(os.path.dirname(args.out) or ".", exist_ok=True)
    with open(args.out, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["path", "label", "class_name", "source_key", "split"])
        w.writeheader()
        w.writerows(rows)

    n_tr = sum(r["split"] == "train" for r in rows)
    n_va = sum(r["split"] == "val" for r in rows)
    n_keys = len({r["source_key"] for r in rows})
    print(f"{len(rows)} files  {n_keys} source keys  ->  train {n_tr} / val {n_va}")
    by_cls: dict[str, int] = {}
    for r in rows:
        by_cls[r["class_name"]] = by_cls.get(r["class_name"], 0) + 1
    for c, n in sorted(by_cls.items()):
        print(f"  {n:5d}  {c}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/faultid.py
"""Supervised fault-identity head on frozen embeddings.

Secondary mode. Where a labelled corpus covers a fault family, a small MLP on
the same frozen embedding names the likely fault. Trained and evaluated with a
source-disjoint split (see prepare_engine_sounds.py) so the macro-F1 reflects
transfer to unseen recordings, not memorised ones.

    python src/faultid.py --manifest data/engine_sounds_manifest.csv --backend torch:mn10_as
"""
from __future__ import annotations

import argparse
import csv
import json
import os
import sys

import numpy as np

_HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, _HERE)

import features as F                        # noqa: E402
from embed_backends import get_backend      # noqa: E402


def read_manifest(path: str):
    rows = list(csv.DictReader(open(path, encoding="utf-8")))
    classes = sorted({r["class_name"] for r in rows})
    return rows, {c: i for i, c in enumerate(classes)}


def embed_split(backend, rows, split, cap=None):
    sel = [r for r in rows if r["split"] == split]
    if cap:
        sel = sel[:cap]
    waves = [F.load_audio(r["path"]) for r in sel]
    emb = backend.embed(waves)
    y = np.array([int(r["label"]) for r in sel])
    return emb, y


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--manifest", required=True)
    ap.add_argument("--backend", default="torch:mn10_as")
    ap.add_argument("--epochs", type=int, default=60)
    ap.add_argument("--cap", type=int, default=None)
    args = ap.parse_args()

    import torch
    import torch.nn as nn
    from sklearn.metrics import f1_score, classification_report

    rows, cls_map = read_manifest(args.manifest)
    backend = get_backend(args.backend)

    Xtr, ytr = embed_split(backend, rows, "train", args.cap)
    Xva, yva = embed_split(backend, rows, "val", args.cap)
    n_cls = len(cls_map)

    clf = nn.Sequential(nn.Linear(Xtr.shape[1], 256), nn.ReLU(), nn.Dropout(0.3),
                        nn.Linear(256, n_cls))
    opt = torch.optim.Adam(clf.parameters(), lr=1e-3, weight_decay=1e-4)
    lossf = nn.CrossEntropyLoss()
    Xt, yt = torch.from_numpy(Xtr).float(), torch.from_numpy(ytr).long()
    for ep in range(args.epochs):
        clf.train()
        opt.zero_grad()
        loss = lossf(clf(Xt), yt)
        loss.backward()
        opt.step()

    clf.eval()
    with torch.no_grad():
        pred = clf(torch.from_numpy(Xva).float()).argmax(1).numpy()
    macro = f1_score(yva, pred, average="macro")
    print(classification_report(yva, pred, target_names=list(cls_map), zero_division=0))

    res = {"backend": args.backend, "macro_f1": float(macro),
           "n_train": int(len(ytr)), "n_val": int(len(yva)), "classes": list(cls_map)}
    out = os.path.join(_HERE, os.pardir, "models",
                       f"faultid_{args.backend.replace(':', '_').replace('/', '_')}.json")
    json.dump(res, open(out, "w"), indent=2)
    print(json.dumps(res, indent=2))
    print("wrote", out)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile export/quantize.py
"""Static QDQ INT8 quantisation of the embedder ONNX graph.

Static, not dynamic: the embedder is almost entirely Conv2d, and ONNX Runtime's
dynamic path only touches MatMul/Gemm, so it would leave the whole network in
FP32. Static QDQ with a real calibration pass over log-mels quantises the
convolutions, which is what the Arm 8-bit dot product accelerates.

Calibration mels come from --calib-dir (any folder of wavs; DCASE train clips or
engine-sounds). Reports FP32 vs INT8 size and a rough MAC count.

    python export/quantize.py --fp32 models/injini_mn10_as_fp32.onnx \
        --calib-dir data/dcase2025_dev/fan/train --n-calib 128
"""
from __future__ import annotations

import argparse
import glob
import json
import os
import sys

import numpy as np
import onnx
import onnxruntime as ort
from onnxruntime.quantization import (
    CalibrationDataReader, CalibrationMethod, QuantFormat, QuantType, quantize_static,
)
from onnxruntime.quantization.preprocess import quant_pre_process

_HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, os.path.join(_HERE, os.pardir, "src"))
import features as F  # noqa: E402


class MelCalib(CalibrationDataReader):
    def __init__(self, wavs, input_name):
        self.wavs = wavs
        self.input_name = input_name
        self.i = 0

    def get_next(self):
        if self.i >= len(self.wavs):
            return None
        mel = F.logmel(F.load_audio(self.wavs[self.i]), fixed=True)
        self.i += 1
        return {self.input_name: mel[None, None, :, :].astype(np.float32)}


def onnx_macs(path: str) -> int:
    """Rough MAC estimate: Conv and Gemm only, from static shapes where present."""
    m = onnx.load(path)
    macs = 0
    init = {i.name for i in m.graph.initializer}
    shapes = {}
    for vi in list(m.graph.value_info) + list(m.graph.input) + list(m.graph.output):
        d = vi.type.tensor_type.shape.dim
        shapes[vi.name] = [x.dim_value if x.dim_value > 0 else 1 for x in d]
    w = {i.name: list(i.dims) for i in m.graph.initializer}
    for n in m.graph.node:
        if n.op_type == "Conv":
            wt = next((w[i] for i in n.input if i in w), None)
            out = shapes.get(n.output[0])
            if wt and out and len(wt) == 4 and len(out) == 4:
                cout, cin, kh, kw = wt
                _, _, oh, ow = out
                macs += cout * cin * kh * kw * oh * ow
        elif n.op_type in ("Gemm", "MatMul"):
            wt = next((w[i] for i in n.input if i in w), None)
            if wt and len(wt) == 2:
                macs += wt[0] * wt[1]
    return macs


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--fp32", required=True)
    ap.add_argument("--calib-dir", required=True)
    ap.add_argument("--n-calib", type=int, default=128)
    ap.add_argument("--out", default=None)
    args = ap.parse_args()

    out = args.out or args.fp32.replace("_fp32.onnx", "_int8.onnx")
    pre = args.fp32.replace("_fp32.onnx", "_fp32_preproc.onnx")
    quant_pre_process(args.fp32, pre, skip_symbolic_shape=True)

    wavs = sorted(glob.glob(os.path.join(args.calib_dir, "**", "*.wav"), recursive=True))
    if not wavs:
        raise SystemExit(f"no calibration wavs under {args.calib_dir}")
    rng = np.random.RandomState(42)
    wavs = list(rng.choice(wavs, size=min(args.n_calib, len(wavs)), replace=False))

    iname = ort.InferenceSession(pre, providers=["CPUExecutionProvider"]).get_inputs()[0].name
    quantize_static(
        pre, out, MelCalib(wavs, iname),
        quant_format=QuantFormat.QDQ,
        activation_type=QuantType.QInt8, weight_type=QuantType.QInt8,
        calibrate_method=CalibrationMethod.MinMax,
        per_channel=True,
    )

    fp32_mb = os.path.getsize(args.fp32) / 1e6
    int8_mb = os.path.getsize(out) / 1e6
    report = {
        "fp32_onnx": args.fp32, "int8_onnx": out,
        "fp32_mb": round(fp32_mb, 3), "int8_mb": round(int8_mb, 3),
        "size_ratio": round(fp32_mb / int8_mb, 2),
        "approx_macs": onnx_macs(args.fp32),
        "n_calibration": len(wavs),
        "quant": "static QDQ INT8, per-channel weights, MinMax",
    }
    rp = out.replace(".onnx", "_quant_report.json")
    json.dump(report, open(rp, "w"), indent=2)
    print(json.dumps(report, indent=2))
    print("wrote", rp)


if __name__ == "__main__":
    main()

## 2. Dependencies, EfficientAT code and weights

In [ ]:

!pip -q install onnx onnxruntime 2>/dev/null
!pip -q install hear21passt 2>/dev/null || echo "hear21passt install failed; PaSST reference row will be skipped"
try:
    import torchaudio  # noqa: F401
except Exception:
    !pip -q install torchaudio 2>/dev/null
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())

if not os.path.isdir("vendor_efficientat"):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/fschmid56/EfficientAT.git", "vendor_efficientat"], check=True)
os.makedirs("vendor_efficientat/resources", exist_ok=True)
for f in ["mn10_as_mAP_471.pt", "mn04_as_mAP_432.pt"]:
    p = f"vendor_efficientat/resources/{f}"
    if not os.path.exists(p):
        subprocess.run(["wget", "-q", "-O", p,
                        f"https://github.com/fschmid56/EfficientAT/releases/download/v0.0.1/{f}"], check=True)
print(os.listdir("vendor_efficientat/resources"))

# cache the Kaldi mel matrix (needs torchaudio, present on Kaggle)
import features as F
F.dump_mel_matrix()
print("mel matrix", os.path.exists("models/mel_kaldi_128x513.npy"))

## 3. DCASE 2025 Task 2 development set (Zenodo 15097779, CC BY-NC-SA 4.0)

In [ ]:

t0 = time.time()
!python src/fetch_dcase.py --out data/dcase2025_dev
n = len(glob.glob("data/dcase2025_dev/*/*/*.wav"))
print(f"{n} wav files in {time.time()-t0:.0f}s")
assert n > 1000, "DCASE dev set did not download"

## 4. Reproduce the DCASE autoencoder baseline

In [ ]:
!python src/baseline_ae.py --root data/dcase2025_dev --epochs 100

## 5. Reference ceiling — PaSST transformer embedding + Mahalanobis
The published state of the art in one reproducible form.

In [ ]:

!python src/eval_dcase.py --root data/dcase2025_dev --backend passt --scorer maha     --out models/eval_passt_maha.json || echo "passt backend failed, continuing"

## 6. Frozen EfficientAT mn10_as — FP32, then static INT8

In [ ]:

!python src/embedder.py --name mn10_as --out models/injini_mn10_as_fp32.onnx
!python src/eval_dcase.py --root data/dcase2025_dev     --backend onnx:models/injini_mn10_as_fp32.onnx --scorer maha --out models/eval_mn10_fp32_maha.json
!python export/quantize.py --fp32 models/injini_mn10_as_fp32.onnx --calib-dir data/dcase2025_dev --n-calib 256
!python src/eval_dcase.py --root data/dcase2025_dev     --backend onnx:models/injini_mn10_as_int8.onnx --scorer maha --out models/eval_mn10_int8_maha.json

## 7. Smaller candidate — mn04_as, FP32 and INT8

In [ ]:

!python src/embedder.py --name mn04_as --out models/injini_mn04_as_fp32.onnx
!python src/eval_dcase.py --root data/dcase2025_dev     --backend onnx:models/injini_mn04_as_fp32.onnx --scorer maha --out models/eval_mn04_fp32_maha.json
!python export/quantize.py --fp32 models/injini_mn04_as_fp32.onnx --calib-dir data/dcase2025_dev --n-calib 256
!python src/eval_dcase.py --root data/dcase2025_dev     --backend onnx:models/injini_mn04_as_int8.onnx --scorer maha --out models/eval_mn04_int8_maha.json

## 8. kNN scorer cross-check (mn10_as FP32)

In [ ]:

!python src/eval_dcase.py --root data/dcase2025_dev     --backend onnx:models/injini_mn10_as_fp32.onnx --scorer knn --out models/eval_mn10_fp32_knn.json

## 9. Supervised fault-ID head (Kaggle engine-sounds, source-disjoint)

In [ ]:

ES = "/kaggle/input/engine-sounds"
!python src/prepare_engine_sounds.py --root {ES} --out data/engine_sounds_manifest.csv
!python src/faultid.py --manifest data/engine_sounds_manifest.csv     --backend onnx:models/injini_mn10_as_fp32.onnx --epochs 80

## 10. Collect the table and export artefacts

In [ ]:

rows = []
for p in sorted(glob.glob("models/eval_*.json")) + sorted(glob.glob("models/faultid_*.json")):
    d = json.load(open(p))
    rows.append({"file": os.path.basename(p),
                 "backend": d.get("backend") or d.get("system"),
                 "scorer": d.get("scorer"),
                 "official_score": d.get("official_score"),
                 "mean_auc": d.get("mean_auc"),
                 "macro_f1": d.get("macro_f1"),
                 "per_machine": d.get("per_machine")})
quant = {os.path.basename(p): json.load(open(p)) for p in glob.glob("models/*_quant_report.json")}
summary = {"generated": time.strftime("%Y-%m-%d %H:%M UTC", time.gmtime()), "results": rows, "quant": quant}
json.dump(summary, open("/kaggle/working/injini_metrics.json", "w"), indent=2)

for r in rows:
    print(f"{r['file']:32s} official={r['official_score']}  mean_auc={r['mean_auc']}  macro_f1={r['macro_f1']}")
print()
for k, v in quant.items():
    print(k, {kk: v.get(kk) for kk in ("fp32_mb", "int8_mb", "size_ratio", "approx_macs")})

for f in glob.glob("models/injini_*.onnx") + glob.glob("models/*_quant_report.json") + glob.glob("models/eval_*.json"):
    shutil.copy(f, "/kaggle/working/")
print("\nworking:", sorted(os.listdir("/kaggle/working")))